# LunarSite Stage 1 — Crater Detection (DeepMoon, v1 seed 1)

Binary U-Net segmentation on DeepMoon 256x256 DEM tiles. Matches the Stage 2 v1 pattern:
U-Net + ResNet-34 (imagenet) + Dice+CE, AdamW, cosine LR, mixed precision, flip-TTA at test time.

- Input: 1-channel DEM (256x256)
- Output: binary crater-ring mask
- Dataset: Silburt 2019 DeepMoon (Zenodo 1133969) — downloaded at runtime from Zenodo.
- Seed controlled by `SEED` constant (default 1). Bump for ensemble members.

In [ ]:
# Pin torch to match Stage 2 ensemble fix (Kaggle P100 sm_60 support).
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.5.1 torchvision==0.20.1
!pip install -q segmentation-models-pytorch h5py albumentations

In [ ]:
import os, sys, json, time, random, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import h5py
import albumentations as A
import segmentation_models_pytorch as smp

print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('smp', smp.__version__)

In [ ]:
# ================== CONFIG ==================
SEED = 1                 # bump for ensemble members
EPOCHS = 40
BATCH_SIZE = 16
LR = 1e-4
WD = 1e-4
ETA_MIN = 1e-6
NUM_WORKERS = 2
IMG_SIZE = 256

WORK = Path('/kaggle/working')
CKPT_PATH = WORK / f'best_craterunet_seed{SEED}.pt'
SUMMARY_PATH = WORK / f'summary_seed{SEED}.json'

TRAIN_H5 = WORK / 'train_images.hdf5'
DEV_H5   = WORK / 'dev_images.hdf5'
TEST_H5  = WORK / 'test_images.hdf5'

ZENODO = 'https://zenodo.org/records/1133969/files'
FILES = {
    TRAIN_H5: f'{ZENODO}/train_images.hdf5',
    DEV_H5:   f'{ZENODO}/dev_images.hdf5',
    TEST_H5:  f'{ZENODO}/test_images.hdf5',
}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, 'seed:', SEED)

In [ ]:
# ================== DOWNLOAD DEEPMOON ==================
# Streaming download with `if not exists` guard so re-runs are fast.
import requests

def download(url: str, dest: Path, chunk=1 << 20):
    if dest.exists() and dest.stat().st_size > 0:
        print(f'exists: {dest.name} ({dest.stat().st_size/1e9:.2f} GB)'); return
    print(f'downloading {url} -> {dest}')
    t0 = time.time()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        dl = 0
        with open(dest, 'wb') as f:
            for c in r.iter_content(chunk_size=chunk):
                if c:
                    f.write(c); dl += len(c)
                    if total and (dl % (200 * chunk) < chunk):
                        print(f'  {dl/1e9:.2f}/{total/1e9:.2f} GB  ({dl/total*100:.1f}%)  {(time.time()-t0)/60:.1f}m')
    print(f'done {dest.name} in {(time.time()-t0)/60:.1f} min')

for dest, url in FILES.items():
    download(url, dest)

# Quick schema check.
with h5py.File(TRAIN_H5, 'r') as f:
    print('train input_images:', f['input_images'].shape, f['input_images'].dtype)
    print('train target_masks:', f['target_masks'].shape, f['target_masks'].dtype)

In [ ]:
# ================== DATASET ==================
# Inline port of src/lunarsite/data/crater_dataset.py (Kaggle has no package install).

class DeepMoonCraterDataset(Dataset):
    def __init__(self, hdf5_path, transform=None, binary_threshold=0.0):
        self.hdf5_path = str(hdf5_path)
        self.transform = transform
        self.binary_threshold = binary_threshold
        self._file = None
        with h5py.File(self.hdf5_path, 'r') as f:
            self.length = f['input_images'].shape[0]

    def _open(self):
        if self._file is None:
            self._file = h5py.File(self.hdf5_path, 'r', swmr=True)
        return self._file

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        f = self._open()
        img = f['input_images'][idx][...]    # (256, 256) uint8
        mask = f['target_masks'][idx][...]   # (256, 256) float32 in [0, 1]
        mask_bin = (mask > self.binary_threshold).astype(np.uint8)
        if self.transform is not None:
            t = self.transform(image=img, mask=mask_bin)
            img, mask_bin = t['image'], t['mask']
        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)   # (1,H,W)
        mask_t = torch.from_numpy(mask_bin.astype(np.float32))                    # (H,W) float
        return {'image': img_t, 'mask': mask_t}

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])
# val/test: no aug

train_ds = DeepMoonCraterDataset(TRAIN_H5, transform=train_aug)
val_ds   = DeepMoonCraterDataset(DEV_H5,   transform=None)
test_ds  = DeepMoonCraterDataset(TEST_H5,  transform=None)
print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

In [ ]:
# ================== LOADERS ==================
def worker_init_fn(worker_id):
    s = SEED + worker_id
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          worker_init_fn=worker_init_fn, generator=g, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# ================== MODEL / LOSS / OPT ==================
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=1,
    classes=1,
).to(device)

dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss = nn.BCEWithLogitsLoss()

def criterion(logits, target):
    # logits: (B,1,H,W), target: (B,H,W)
    t = target.unsqueeze(1)
    return 0.5 * dice_loss(logits, t) + 0.5 * bce_loss(logits, t)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=ETA_MIN)
scaler = GradScaler('cuda')

n_params = sum(p.numel() for p in model.parameters())
print(f'params: {n_params/1e6:.2f}M')

In [ ]:
# ================== METRICS ==================
@torch.no_grad()
def iou_dice_from_logits(logits, target, thresh=0.5, eps=1e-7):
    # logits (B,1,H,W), target (B,H,W) in {0,1}
    pred = (torch.sigmoid(logits).squeeze(1) > thresh).float()
    tgt = target.float()
    inter = (pred * tgt).sum(dim=(1,2))
    union = pred.sum(dim=(1,2)) + tgt.sum(dim=(1,2)) - inter
    iou = (inter + eps) / (union + eps)
    dice = (2 * inter + eps) / (pred.sum(dim=(1,2)) + tgt.sum(dim=(1,2)) + eps)
    return iou, dice

@torch.no_grad()
def evaluate(model, loader, tta=False):
    model.eval()
    ious, dices = [], []
    for batch in loader:
        x = batch['image'].to(device, non_blocking=True)
        y = batch['mask'].to(device, non_blocking=True)
        with autocast('cuda'):
            if not tta:
                logits = model(x)
            else:
                # 4-way flip TTA: id, hflip, vflip, hvflip. Average sigmoid probs.
                probs = torch.zeros_like(x)  # (B,1,H,W)
                for hflip in (False, True):
                    for vflip in (False, True):
                        xi = x
                        if hflip: xi = torch.flip(xi, dims=[-1])
                        if vflip: xi = torch.flip(xi, dims=[-2])
                        li = model(xi)
                        pi = torch.sigmoid(li)
                        if vflip: pi = torch.flip(pi, dims=[-2])
                        if hflip: pi = torch.flip(pi, dims=[-1])
                        probs = probs + pi
                probs = probs / 4.0
                # convert average prob back to 'logits-like' for the threshold path
                logits = torch.log(probs.clamp(1e-7, 1-1e-7) / (1 - probs.clamp(1e-7, 1-1e-7)))
        iou, dice = iou_dice_from_logits(logits.float(), y)
        ious.append(iou.cpu()); dices.append(dice.cpu())
    return float(torch.cat(ious).mean()), float(torch.cat(dices).mean())

In [ ]:
# ================== TRAIN ==================
best_val_iou = -1.0
best_val_dice = -1.0
best_epoch = -1
history = []
t_train0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss, n = 0.0, 0
    t0 = time.time()
    for batch in train_loader:
        x = batch['image'].to(device, non_blocking=True)
        y = batch['mask'].to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda'):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        ep_loss += float(loss.item()) * x.size(0); n += x.size(0)
    sched.step()
    train_loss = ep_loss / max(n, 1)

    val_iou, val_dice = evaluate(model, val_loader, tta=False)
    dt = time.time() - t0
    lr_now = opt.param_groups[0]['lr']
    print(f'epoch {epoch:02d}/{EPOCHS}  loss {train_loss:.4f}  val_iou {val_iou:.4f}  val_dice {val_dice:.4f}  lr {lr_now:.2e}  ({dt/60:.1f}m)')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_iou': val_iou, 'val_dice': val_dice, 'lr': lr_now})

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_val_dice = val_dice
        best_epoch = epoch
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_iou': val_iou, 'val_dice': val_dice, 'seed': SEED}, CKPT_PATH)
        print(f'  -> saved new best to {CKPT_PATH}')

print(f'training done in {(time.time()-t_train0)/60:.1f} min. best val_iou {best_val_iou:.4f} @ epoch {best_epoch}')

In [ ]:
# ================== TEST (best checkpoint) ==================
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])

test_iou, test_dice = evaluate(model, test_loader, tta=False)
print(f'test (no TTA):   iou {test_iou:.4f}  dice {test_dice:.4f}')

test_tta_iou, test_tta_dice = evaluate(model, test_loader, tta=True)
print(f'test (flip TTA): iou {test_tta_iou:.4f}  dice {test_tta_dice:.4f}')

In [ ]:
summary = {
    'seed': SEED,
    'best_val_iou': best_val_iou,
    'best_val_dice': best_val_dice,
    'best_epoch': best_epoch,
    'test_iou': test_iou,
    'test_dice': test_dice,
    'test_tta_iou': test_tta_iou,
    'test_tta_dice': test_tta_dice,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'wd': WD,
    'img_size': IMG_SIZE,
    'arch': 'smp.Unet resnet34 imagenet in1 cls1',
    'loss': '0.5 DiceLoss(binary) + 0.5 BCEWithLogits',
    'history': history,
}
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps({k: v for k, v in summary.items() if k != 'history'}, indent=2))
print(f'summary -> {SUMMARY_PATH}')
print(f'ckpt    -> {CKPT_PATH}')